In [ ]:
# AI Assisted code 
# Code generated by Claude
# Adapted by authors

import pandas as pd
import requests

# List of countries
countries = [
    "Norway",
    "Sweden",
    "Portugal",
    "Spain",
    "Brazil",
    "Canada",
    "Austria",
    "Switzerland",
    "Germany",
    "France",
    "Italy",
    "Netherlands",
    "United Kingdom",
    "United States"
]

start_year = 2005
end_year = 2023

country_codes = [
    "NOR", "SWE", "PRT", "ESP",
    "BRA", "CAN", "AUT", "CHE",
    "DEU", "FRA", "ITA", "NLD",
    "GBR", "USA"
]

country_string = ";".join(country_codes)

url = (
    f"https://api.worldbank.org/v2/"
    f"country/{country_string}/"
    f"indicator/EN.CLC.SPEI.XD"
    f"?date={start_year}:{end_year}"
    f"&format=json"
    f"&per_page=1000"
)

response = requests.get(url)
response.raise_for_status()

data = response.json()

records = []

for item in data[1]:
    records.append({
        "country": item["country"]["value"],
        "country_code": item["countryiso3code"],
        "year": int(item["date"]),
        "spei_12": item["value"]
    })

drought_df = pd.DataFrame(records)

# Drought definition
drought_df["drought"] = drought_df["spei_12"] <= -1

energy_url = (
    "https://raw.githubusercontent.com/"
    "owid/energy-data/master/owid-energy-data.csv"
)

energy_df = pd.read_csv(energy_url)

energy_clean = energy_df[
    (energy_df["country"].isin(countries)) &
    (energy_df["year"].between(start_year, end_year))
].copy()

energy_columns = [
    "country",
    "year",
    "electricity_generation",

    # Renewables
    "hydro_electricity",
    "wind_electricity",
    "solar_electricity",
    "biofuel_electricity",

    # Nuclear
    "nuclear_electricity",

    # Fossil fuels
    "coal_electricity",
    "gas_electricity",
    "oil_electricity",
    "fossil_electricity",

    # Electricity trade
    "net_elec_imports",
    "net_elec_imports_share_demand"
]

energy_clean = energy_clean[energy_columns]

# Merge drought data with energy data on country and year
final_df = pd.merge(
    drought_df,
    energy_clean,
    on=["country", "year"],
    how="inner"
)

energy_sources = [
    "hydro",
    "wind",
    "solar",
    "biofuel",
    "nuclear",
    "coal",
    "gas",
    "oil",
    "fossil"
]

for source in energy_sources:
    final_df[f"{source}_share"] = (
        final_df[f"{source}_electricity"]
        / final_df["electricity_generation"]
    )

final_df = final_df.sort_values(
    ["country", "year"]
).reset_index(drop=True)

print("Shape:", final_df.shape)

print("\nCountries:")
print(final_df["country"].unique())

print("\nYears:")
print(final_df["year"].min(), "-", final_df["year"].max())

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nDuplicates:")
print(
    final_df.duplicated(
        subset=["country", "year"]
    ).sum()
)

print("\nFinal columns:")
print(final_df.columns.tolist())


In [ ]:
final_df.to_csv(
    "drought_energy_clean.csv",
    index=False
)

print("Saved: drought_energy_clean.csv")
final_df.head()
